# Section 3 — Drift and diffusion

<!-- GENERATED by logistics/scripts/stage_section_pages.py from sections/sec03/handout.md. Do not edit here; edit the section file. -->

A walker takes a step $\delta$ every $\tau$: right with probability $p$, left with $q = 1-p$. Two numbers summarize what it does. It drifts at
$$
v = \frac{(p-q)\delta}{\tau},
$$
and it spreads by $\sqrt{2Dt}$, with
$$
D = \frac{\delta^{2}}{2\tau}.
$$
One of those grows in proportion to $t$ and the other does not, which is the whole content of this page.

*Agents are out for the warm-up. Do it on paper, then come back here and check it.*

:::{note} Running the code
The Python cells on this page run **in your browser**. Click the **power icon** at the top of the page to activate the kernel, then run — or edit — any cell. Packages load automatically the first time you import them (a few seconds).
:::

## The warm-up

A small molecule in water has $D = 500\ \mu\mathrm{m}^2/\mathrm{s}$, and the water around it moves at $v = 10\ \mu\mathrm{m/s}$.

1. After what time, and over what distance, does being carried match spreading out?
2. A cell is about $10\ \mu$m across, a leaf about 1 mm thick. Which one can let things spread on their own?

*AM215: both answers to question 1 are built out of $D$ and $v$ alone. Show why, from units.*

## Many walkers make a Gaussian

Take $\delta = 1$ and $\tau = 1$, so $v = 0$ and $D = 1/2$ and the predicted density after $n$ steps is a Gaussian of variance $2Dn = n$.

The number of right-steps in $n$ tries is binomial, so a walker's final position is $2k - n$ for $k \sim \mathrm{Binomial}(n, p)$. Drawing $k$ directly is exact and costs one number per walker, which is why the cell below can afford 200,000 of them.

One detail matters when a lattice is compared with a continuum: after $n$ steps a walker can only sit on sites of the same parity as $n$, spaced $2\delta$ apart. The bin width below is therefore $2\delta$, and that factor of $2\delta$ is the one the derivation divides out when it writes $P_n(x) \approx 2\delta\,\rho(x, n\tau)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 12, "figure.figsize": (7, 4.2)})

delta, tau = 1.0, 1.0
D = delta**2 / (2 * tau)
rng = np.random.default_rng(seed=215)


def final_positions(n_steps, n_walkers, p, rng, delta=1.0):
    """Positions of n_walkers independent walkers after n_steps steps.

    The number of right-steps is Binomial(n_steps, p), so the position is
    (2k - n_steps) * delta. Exact, and O(n_walkers) in memory.
    """
    k = rng.binomial(n_steps, p, size=n_walkers)
    return (2 * k - n_steps) * delta


def gaussian_solution(x, t, D, v=0.0):
    """The fundamental solution G_v(x, t) of the drift-diffusion equation."""
    return np.exp(-((x - v * t) ** 2) / (4 * D * t)) / np.sqrt(4 * np.pi * D * t)


fig, ax = plt.subplots()
for n_steps, color in [(100, "tab:blue"), (400, "tab:orange"), (1600, "tab:green")]:
    X = final_positions(n_steps, 200_000, p=0.5, rng=rng, delta=delta)
    lim = 4 * np.sqrt(2 * D * n_steps)
    edges = np.arange(-lim, lim + 2 * delta, 2 * delta) - delta
    ax.hist(X, bins=edges, density=True, color=color, alpha=0.35)

    x = np.linspace(-lim, lim, 400)
    ax.plot(x, gaussian_solution(x, n_steps * tau, D), color=color, lw=2,
            label=f"$G(x,t)$, $t={n_steps}$")

ax.set_xlabel("position $x$")
ax.set_ylabel(r"density $\rho(x,t)$")
ax.set_title("Walkers (bars) against the solution of the diffusion equation (lines)")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

The curves are not fitted to the bars. $D = \delta^2/2\tau$ was read off the walk rule and $G$ was evaluated at it. Agreement here is **verification**: the simulation and the formula describe the same model, and they agree. It says nothing yet about anything that really moves.

## The variance grows at $2D$

The Gaussian hides a sharper claim. The variance is linear in $t$ with slope $2D$, which is what gives $D$ its units of length$^2$/time.

In [ ]:
times = np.array([25, 50, 100, 200, 400, 800, 1600])
variances = np.array(
    [final_positions(n, 40_000, p=0.5, rng=rng, delta=delta).var() for n in times]
)
slope = np.polyfit(times, variances, 1)[0]

print(f"fitted slope   = {slope:.4f}")
print(f"2D (predicted) = {2 * D:.4f}")

# fitted slope   = 1.0065
# 2D (predicted) = 1.0000

## How long can a bias hide?

Take $p = 0.55$, so the walker really is biased, and compare the two things it does. Setting $vt = \sqrt{2Dt}$ gives
$$
t^{*} = \frac{2D}{v^{2}}, \qquad \ell = v\,t^{*} = \frac{2D}{v}.
$$
Before $t^{*}$ the bias sits inside the noise; after it, the bias is the whole story.

In [ ]:
p = 0.55
v = (p - (1 - p)) * delta / tau
t_star = 2 * D / v**2

print(f"v = {v:.3f}   D = {D:.3f}   t* = {t_star:.0f} steps   l = {2 * D / v:.0f}")
for n in (25, 50, 100, 400):
    X = final_positions(n, 40_000, p=p, rng=rng, delta=delta)
    print(f"t = {n:4d}:  mean = {X.mean():7.2f}   sd = {X.std():6.2f}"
          f"   drift/spread = {X.mean() / X.std():.2f}")

# v = 0.100   D = 0.500   t* = 100 steps   l = 10
# t =   25:  mean =    2.49   sd =   4.99   drift/spread = 0.50
# t =   50:  mean =    5.03   sd =   7.08   drift/spread = 0.71
# t =  100:  mean =    9.96   sd =   9.98   drift/spread = 1.00
# t =  400:  mean =   40.12   sd =  19.84   drift/spread = 2.02

A walker that wins 55% of its steps is still indistinguishable from a fair one after 50 of them: the drift is 5 and the spread is 7. Seeing an edge of size $\epsilon$ takes of order $1/\epsilon^{2}$ steps, which is the $1/\sqrt N$ law read backwards.

Two versions of $t^{*}$ are worth keeping apart. On the lattice the exact answer is $4pq/(p-q)^2 = 99$, because the one-step variance is $1 - (p-q)^2$ rather than 1. The continuum coefficient $D = \delta^2/2\tau$ gives 100. The gap is the $1\%$ that the limit $p - q \to 0$ removes.

In [ ]:
fig, ax = plt.subplots()
t = np.linspace(1, 400, 400)
ax.plot(t, v * t, label=r"drift $vt$")
ax.plot(t, np.sqrt(2 * D * t), label=r"spread $\sqrt{2Dt}$")
ax.axvline(t_star, color="k", ls=":", lw=1)
ax.annotate(rf"$t^*={t_star:.0f}$", xy=(t_star, v * t_star),
            xytext=(t_star + 25, v * t_star * 0.55))
ax.set_xlabel("time $t$")
ax.set_ylabel("distance")
ax.set_title("A small bias is invisible until the drift outgrows the spread")
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## Checking the warm-up

In [ ]:
D_water, v_water = 500.0, 10.0          # um^2/s, um/s
print(f"t* = {2 * D_water / v_water**2:.0f} s")
print(f"l  = {2 * D_water / v_water:.0f} um")

for name, L in [("cell", 10.0), ("leaf", 1000.0)]:
    print(f"{name:5s} L = {L:6.0f} um:  spreading L^2/2D = {L**2 / (2 * D_water):8.2f} s"
          f"   carried L/v = {L / v_water:7.2f} s")

# t* = 10 s
# l  = 100 um
# cell  L =     10 um:  spreading L^2/2D =     0.10 s   carried L/v =    1.00 s
# leaf  L =   1000 um:  spreading L^2/2D =  1000.00 s   carried L/v =  100.00 s

A 10 μm cell sits well inside $\ell = 100\ \mu$m. Spreading crosses it in a tenth of a second, so nothing there has to be pumped. A 1 mm leaf is ten times outside it, and spreading alone would need about a thousand seconds, so a leaf moves things. The threshold is not a fitted constant; it is the only length that can be built out of $D$ and $v$.

## Where the derivation is

The derivation behind all of this — the master equation, the Taylor expansion, the fundamental solution, and the biased case as an exercise — is in [Chapter 5: Random Walks & the Diffusion Equation](05_random_walks_diffusion.md).